# 03: Embeddings and RAG Pipeline

**Educational Series: RAG for Financial Documents**

This notebook teaches you how embeddings work and how to build a complete end-to-end RAG (Retrieval-Augmented Generation) pipeline to answer questions about financial documents with citations.

## Section 1: Setup & Imports

### What We're Building

In this notebook, we'll construct a complete RAG pipeline that takes a user question and returns an answer backed by your document chunks. Here's the architecture:

```
Question → Embed → [Qdrant Search] → Top-K Chunks → GPT-4o-mini → Answer with Citations
                         ↑
              [Offline: chunks → embed → index]
```

The pipeline has two phases:
1. **Indexing (offline)**: Convert document chunks into embeddings and store them in Qdrant
2. **Retrieval (online)**: When a user asks a question, embed it, search Qdrant, retrieve top-K chunks, and pass them to an LLM

This allows us to:
- Answer questions accurately using your specific documents
- Cite sources (which chunk each fact came from)
- Keep costs reasonable (don't send entire documents to GPT, only relevant chunks)

In [ ]:
# Standard library imports
import os
import sys
from pathlib import Path
from typing import List
import json
from datetime import datetime

# Third-party imports
import numpy as np
from tqdm import tqdm

# LLM and embeddings
from openai import OpenAI

# Vector database
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct

# LangChain (we'll use this in later sections)
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# Load environment variables
from dotenv import load_dotenv

# Add project root to path for imports
project_root = Path().cwd().parent.parent if 'notebooks' in str(Path().cwd()) else Path()
sys.path.insert(0, str(project_root))

# Load .env file
load_dotenv()

print("✓ Imports successful")
print(f"✓ Project root: {project_root}")
print(f"✓ OpenAI API key configured: {bool(os.getenv('OPENAI_API_KEY'))}")

## Section 2: What is an Embedding?

### Embeddings as Semantic Coordinates

An **embedding** is a list of numbers (a vector) that represents the meaning of a piece of text. Think of it as coordinates in a high-dimensional "meaning space":

- **Similar meaning = nearby points**: Two texts about revenue growth will have embeddings close to each other
- **Different meaning = far points**: A text about employees and one about revenue will have embeddings far apart
- **Dimensionality**: OpenAI's `text-embedding-3-small` produces **1536 numbers** per text (1536-dimensional space)

### Why Cosine Similarity?

We use **cosine similarity** (not Euclidean distance) because:
1. Embeddings are normalized (magnitude ≈ 1)
2. Cosine similarity measures angle, not distance
3. It's more robust to document length variations
4. Ranges from -1 (opposite) to 1 (identical), with 0 = perpendicular

### Visual Intuition

```
   "Revenue grew 18%" ●
                      │  ← close (similar)
  "Sales increased" ●

  "Hired 500 employees" ◐ (far away, different topic)
```

In [ ]:
# Conceptual demo: simulating embeddings to illustrate the idea
# In reality, embeddings would come from OpenAI's API

# Sample financial texts
texts = [
    "Revenue grew 18% year-over-year",
    "Sales increased significantly this quarter", 
    "The company hired 500 new employees",
]

# Simulate embeddings: in reality these would be 1536-dimensional
# For demo, using 10 dimensions for visualization
np.random.seed(42)

# Create vectors that demonstrate semantic similarity
v1 = np.random.randn(10)  # "Revenue grew 18%"
v2 = v1 + np.random.randn(10) * 0.1  # "Sales increased" - close to v1 (similar topic)
v3 = np.random.randn(10)  # "Hired employees" - random (different topic)

# Normalize vectors (as embeddings are normalized)
v1 = v1 / np.linalg.norm(v1)
v2 = v2 / np.linalg.norm(v2)
v3 = v3 / np.linalg.norm(v3)

def cosine_similarity(a, b):
    """Compute cosine similarity between two vectors."""
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

print("=" * 60)
print("COSINE SIMILARITY DEMO")
print("=" * 60)
print(f"\nTexts:")
for i, text in enumerate(texts, 1):
    print(f"  {i}. {text}")

sim_12 = cosine_similarity(v1, v2)
sim_13 = cosine_similarity(v1, v3)
sim_23 = cosine_similarity(v2, v3)

print(f"\nSimilarity Scores (higher = more similar):")
print(f"  Text 1 vs Text 2 (both about revenue/sales):  {sim_12:.3f} ← HIGH (similar topics)")
print(f"  Text 1 vs Text 3 (different topics):          {sim_13:.3f} ← LOW (different topics)")
print(f"  Text 2 vs Text 3 (different topics):          {sim_23:.3f} ← LOW (different topics)")

print(f"\n💡 Key insight: Similar financial topics get high scores!")
print(f"   This is how RAG finds relevant chunks for your question.")

## Section 3: OpenAI Embeddings

### text-embedding-3-small

OpenAI provides several embedding models:

| Model | Dimensions | Cost per 1M tokens | Speed | Use Case |
|-------|-----------|-------------------|-------|----------|
| text-embedding-3-small | 1536 | $0.02 | Fast | Default (our choice) |
| text-embedding-3-large | 3072 | $0.13 | Slower | High-precision |
| text-embedding-ada-002 | 1536 | $0.10 | Slow | Legacy (deprecated) |

### Cost Calculation

- Average chunk: ~100 tokens
- 1000 chunks: 100,000 tokens
- Cost: (100,000 / 1,000,000) × $0.02 = **$0.002** (2/10 of a cent!)

### Batching for Efficiency

Always batch your embedding requests:
- Single requests are slow (one API call per chunk)
- Batched requests (100+ at once) are much faster
- OpenAI's API accepts up to 2,048 input strings per request

In [ ]:
# How to call OpenAI's embedding API

def embed_single_text(text: str, api_key: str, model: str = "text-embedding-3-small") -> List[float]:
    """
    Embed a single text string.
    
    Args:
        text: Text to embed
        api_key: OpenAI API key
        model: Embedding model to use
        
    Returns:
        List of floats (1536 dimensions for text-embedding-3-small)
    """
    client = OpenAI(api_key=api_key)
    response = client.embeddings.create(
        input=text,
        model=model
    )
    return response.data[0].embedding


def embed_batch(texts: List[str], api_key: str, batch_size: int = 100, model: str = "text-embedding-3-small") -> List[List[float]]:
    """
    Embed multiple texts efficiently with batching.
    
    Args:
        texts: List of texts to embed
        api_key: OpenAI API key
        batch_size: Number of texts per API call (max 2048)
        model: Embedding model to use
        
    Returns:
        List of embeddings (each is a list of 1536 floats)
    """
    client = OpenAI(api_key=api_key)
    embeddings = []
    
    # Process in batches
    for i in tqdm(range(0, len(texts), batch_size), desc="Embedding batches"):
        batch = texts[i:i + batch_size]
        response = client.embeddings.create(
            input=batch,
            model=model
        )
        # Each response.data[j].embedding is a list of 1536 floats
        embeddings.extend([r.embedding for r in response.data])
    
    return embeddings


# Cost estimation
def estimate_embedding_cost(num_chunks: int, avg_tokens_per_chunk: int = 100) -> float:
    """Estimate cost of embedding chunks with text-embedding-3-small."""
    total_tokens = num_chunks * avg_tokens_per_chunk
    cost_per_1m_tokens = 0.02
    return (total_tokens / 1_000_000) * cost_per_1m_tokens


# Example usage (would need OPENAI_API_KEY to actually run)
print("=" * 60)
print("OPENAI EMBEDDING API")
print("=" * 60)

print(f"\nModel: text-embedding-3-small")
print(f"Output dimensions: 1536")
print(f"Cost: $0.02 per 1M tokens")

# Cost examples
chunk_counts = [100, 500, 1000, 5000]
print(f"\nCost estimates (at ~100 tokens per chunk):")
for count in chunk_counts:
    cost = estimate_embedding_cost(count)
    print(f"  {count:5d} chunks → ${cost:.6f} (${cost*100:.4f} cents)")

print(f"\n✓ Embeddings are VERY cheap!")
print(f"  1000 chunks costs less than 1 API call to GPT-4o-mini")

## Section 4: Qdrant — Vector Database

### Why We Need a Vector Database

**The naive approach**: Store embeddings in a Python list, linear search all of them
- 100 chunks: ~1ms (fine)
- 10,000 chunks: ~100ms (slow)
- 1M chunks: ~10s (unacceptable)

**Qdrant's approach**: Use vector indexing (HNSW algorithm) for logarithmic search
- 1M chunks: ~5ms (excellent)

### Key Qdrant Concepts

- **Collection**: A group of points with the same vector dimensions (like a table in SQL)
- **Point**: A single vector + optional metadata payload
- **Payload**: Extra data stored with the vector (original text, source file, page number, etc.)
- **Distance metric**: How to measure similarity (we use COSINE)

### Running Qdrant Locally

**Option 1: Docker (Recommended)**

```bash
docker run -p 6333:6333 -p 6334:6334 \
  -v $(pwd)/qdrant_storage:/qdrant/storage:z \
  qdrant/qdrant:latest
```

**Option 2: In-memory (Testing)**

```python
from qdrant_client import QdrantClient
client = QdrantClient(":memory:")
```

In [ ]:
# Example: Creating a Qdrant collection and inserting points

def create_qdrant_collection(client: QdrantClient, collection_name: str, vector_size: int = 1536):
    """
    Create a Qdrant collection for storing embeddings.
    
    Args:
        client: QdrantClient instance
        collection_name: Name for the collection
        vector_size: Dimensionality of embeddings (1536 for text-embedding-3-small)
    """
    client.recreate_collection(
        collection_name=collection_name,
        vectors_config=VectorParams(
            size=vector_size,
            distance=Distance.COSINE  # Use cosine similarity
        ),
    )
    print(f"✓ Created collection: {collection_name}")


def insert_points_to_qdrant(client: QdrantClient, collection_name: str, points: List[PointStruct]):
    """
    Insert points (embeddings + metadata) into a Qdrant collection.
    
    Args:
        client: QdrantClient instance
        collection_name: Name of the collection
        points: List of PointStruct objects
    """
    client.upsert(
        collection_name=collection_name,
        points=points
    )
    print(f"✓ Inserted {len(points)} points into {collection_name}")


# Example with demo data (no real embeddings needed)
print("=" * 60)
print("QDRANT VECTOR DATABASE")
print("=" * 60)

# Create fake vectors for demonstration
fake_vector_1 = [0.1 * i for i in range(1536)]  # Mock 1536-dim vector
fake_vector_2 = [0.2 * i for i in range(1536)]

# Example points with metadata payloads
example_points = [
    PointStruct(
        id=1,
        vector=fake_vector_1,
        payload={
            "text": "Total revenue for fiscal year 2023 reached $15.2 billion, representing an 18% increase.",
            "doc_type": "annual_report",
            "source_file": "2023_annual_report.pdf",
            "page": 5,
            "chunk_index": 0
        }
    ),
    PointStruct(
        id=2,
        vector=fake_vector_2,
        payload={
            "text": "Net income for the period was $3.1 billion with a net margin of 20.4%.",
            "doc_type": "annual_report",
            "source_file": "2023_annual_report.pdf",
            "page": 7,
            "chunk_index": 1
        }
    )
]

print("\nExample: Qdrant Collection Structure")
print(f"\nCollection Config:")
print(f"  - Name: financial_docs")
print(f"  - Vector Size: 1536 dimensions")
print(f"  - Distance Metric: COSINE")

print(f"\nExample Point 1:")
print(f"  ID: {example_points[0].id}")
print(f"  Vector: [0.0, 0.1, 0.2, ..., {1536*0.1:.1f}]  (1536 dims, shown abbreviated)")
print(f"  Payload:")
for key, value in example_points[0].payload.items():
    if key != "text":
        print(f"    {key}: {value}")
    else:
        print(f"    {key}: {value[:60]}...")

print(f"\n✓ Ready to store embeddings and search them efficiently!")

## Section 5: DenseRetriever (From Scratch)

### Building a Retriever Class

Now let's build a `DenseRetriever` class that:
1. Takes document chunks
2. Embeds them with OpenAI
3. Stores them in Qdrant
4. Retrieves top-K relevant chunks for any question

### Key Design Decisions

**Text with headings**: Include the chunk's heading as context in the embedding. This helps the embedding understand the semantic context.

```python
# Instead of:
text = "Revenue grew 18% year-over-year"
embedding = embed(text)

# Do this:
text_with_heading = "## Revenue\nRevenue grew 18% year-over-year"
embedding = embed(text_with_heading)  # Better!
```

**Storing original text in payload**: The embedding is just numbers. Store the original text in Qdrant's payload so we can display it to users and pass it to the LLM.

In [ ]:
# Example usage of a DenseRetriever (conceptual)
# In practice, this would be imported from src.from_scratch.retrieval.dense

class MockDenseRetriever:
    """Simplified retriever for demonstration."""
    
    def __init__(self, collection_name: str, qdrant_url: str, openai_api_key: str):
        self.collection_name = collection_name
        self.qdrant_url = qdrant_url
        self.client = QdrantClient(url=qdrant_url)
        self.openai_client = OpenAI(api_key=openai_api_key)
        
    def index_chunks(self, chunks: List, batch_size: int = 50) -> int:
        """
        Embed chunks and store in Qdrant.
        
        Args:
            chunks: List of chunk objects with .text, .chunk_index, .metadata
            batch_size: Batch size for embedding API calls
            
        Returns:
            Number of chunks indexed
        """
        print(f"Indexing {len(chunks)} chunks...")
        
        # Step 1: Prepare texts (chunk text with heading context)
        texts_to_embed = []
        for chunk in chunks:
            # Include heading context if available
            text_with_context = chunk.text
            if hasattr(chunk, 'metadata') and 'heading' in chunk.metadata:
                text_with_context = f"# {chunk.metadata['heading']}\n{chunk.text}"
            texts_to_embed.append(text_with_context)
        
        # Step 2: Embed in batches
        embeddings = []
        for i in tqdm(range(0, len(texts_to_embed), batch_size), desc="Embedding chunks"):
            batch = texts_to_embed[i:i + batch_size]
            response = self.openai_client.embeddings.create(
                input=batch,
                model="text-embedding-3-small"
            )
            embeddings.extend([r.embedding for r in response.data])
        
        # Step 3: Create PointStruct objects with embeddings + payloads
        points = []
        for i, (chunk, embedding) in enumerate(zip(chunks, embeddings)):
            payload = {
                "text": chunk.text,
                "chunk_index": chunk.chunk_index,
            }
            if hasattr(chunk, 'metadata'):
                payload.update(chunk.metadata)
            
            points.append(PointStruct(
                id=i,
                vector=embedding,
                payload=payload
            ))
        
        # Step 4: Store in Qdrant
        self.client.upsert(
            collection_name=self.collection_name,
            points=points
        )
        
        return len(chunks)
    
    def search(self, question: str, top_k: int = 3):
        """
        Find top-K most relevant chunks for a question.
        
        Args:
            question: User question
            top_k: Number of chunks to retrieve
            
        Returns:
            List of relevant chunks with scores
        """
        # Step 1: Embed the question
        response = self.openai_client.embeddings.create(
            input=question,
            model="text-embedding-3-small"
        )
        question_embedding = response.data[0].embedding
        
        # Step 2: Search Qdrant
        results = self.client.search(
            collection_name=self.collection_name,
            query_vector=question_embedding,
            limit=top_k,
        )
        
        return results


print("=" * 60)
print("DENSE RETRIEVER CLASS")
print("=" * 60)

print("""
✓ DenseRetriever provides two main methods:

1. index_chunks(chunks)
   - Embeds all chunks with OpenAI API
   - Stores embeddings + metadata in Qdrant
   - Returns number of chunks indexed

2. search(question, top_k=3)
   - Embeds the user's question
   - Searches Qdrant for similar chunks
   - Returns top-K results with similarity scores

Example workflow:
  retriever = DenseRetriever(...)
  retriever.index_chunks(doc_chunks)  # Index once
  results = retriever.search("What was the revenue?")  # Search many times
""")

In [ ]:
# Example: Using the retriever with sample financial data

# Sample financial document
sample_text = """
# Annual Financial Report 2023

## Revenue
Total revenue for fiscal year 2023 reached $15.2 billion, representing 
an 18% increase compared to the prior year. This growth was primarily 
driven by strong performance in our cloud services division, which grew 35% year-over-year.

## Operating Expenses  
Operating expenses totaled $9.8 billion, a 12% increase year-over-year.
The increase was driven by headcount growth and infrastructure investments
needed to support our expanding cloud platform.

## Net Income
Net income for the period was $3.1 billion, with a net margin of 20.4%.
This represents a healthy profitability despite increased investments in R&D.

## Cash Flow
Operating cash flow increased to $4.8 billion from $4.2 billion in the prior year.
Capital expenditures increased to $2.1 billion to support infrastructure growth.
"""

# Mock chunk objects (in real project, these come from document chunking)
class MockChunk:
    def __init__(self, text: str, chunk_index: int, metadata: dict):
        self.text = text
        self.chunk_index = chunk_index
        self.metadata = metadata

# Split into chunks (simplified - using fixed size)
chunk_size = 300
chunks = []
for i in range(0, len(sample_text), chunk_size):
    chunk_text = sample_text[i:i + chunk_size]
    chunks.append(MockChunk(
        text=chunk_text,
        chunk_index=i // chunk_size,
        metadata={
            "doc_type": "annual_report",
            "company": "TechCorp Inc.",
            "year": 2023,
            "source_file": "2023_annual_report.pdf"
        }
    ))

print("=" * 60)
print("EXAMPLE: DOCUMENT CHUNKING")
print("=" * 60)
print(f"\nOriginal document: {len(sample_text)} characters")
print(f"Split into: {len(chunks)} chunks (300 chars each)")
print(f"\nChunks created:")
for chunk in chunks:
    preview = chunk.text[:60].replace('\n', ' ') + '...'
    print(f"  [{chunk.chunk_index}] {preview}")

print(f"\n✓ Now these chunks would be:")
print(f"  1. Embedded with OpenAI API")
print(f"  2. Stored in Qdrant with metadata")
print(f"  3. Ready for semantic search!")

In [ ]:
# Retriever usage example (guarded by API key check)

if os.getenv("OPENAI_API_KEY"):
    print("=" * 60)
    print("RETRIEVER IN ACTION (API available)")
    print("=" * 60)
    
    print("""
    # Indexing phase (one-time)
    retriever = DenseRetriever(
        collection_name="financial_2023",
        qdrant_url="http://localhost:6333",
        openai_api_key=os.getenv("OPENAI_API_KEY"),
    )
    n_indexed = retriever.index_chunks(chunks)
    print(f"Indexed {n_indexed} chunks")
    
    # Search phase (run many times)
    results = retriever.search("What was the revenue growth?", top_k=3)
    
    for result in results:
        print(f"\nScore: {result.score:.3f} | Rank: {result.rank}")
        print(f"Text: {result.payload['text'][:100]}...")
        print(f"Source: {result.payload['source_file']} (Page {result.payload.get('page', 'N/A')})")
    """)
    
else:
    print("=" * 60)
    print("RETRIEVER IN ACTION (Expected Output)")
    print("=" * 60)
    print("""
    Note: Set OPENAI_API_KEY environment variable to run indexing.
    
    Expected output from retriever.search("What was the revenue growth?", top_k=3):
    
    Result 1 (Most relevant):
      Score: 0.891 | Rank: 1
      Text: Total revenue for fiscal year 2023 reached $15.2 billion, 
            representing an 18% increase compared to the prior year...
      Source: 2023_annual_report.pdf (Page N/A)
    
    Result 2:
      Score: 0.756 | Rank: 2
      Text: This growth was primarily driven by strong performance in 
            our cloud services division, which grew 35% year-over-year...
      Source: 2023_annual_report.pdf (Page N/A)
    
    Result 3
      Score: 0.634 | Rank: 3
      Text: Operating expenses totaled $9.8 billion, a 12% increase 
            year-over-year...
      Source: 2023_annual_report.pdf (Page N/A)
    """)
    print(f"\n✓ Set OPENAI_API_KEY environment variable to enable API calls")

## Section 6: Generating Answers (From Scratch)

### The Answer Generation Pipeline

Once we have retrieved relevant chunks, we:
1. **Build a context block** with numbered citations
2. **Construct a prompt** with system instructions + context + question
3. **Call GPT-4o-mini** to generate an answer
4. **Parse the answer** and extract cost metrics

### Prompt Design Best Practices

**Use numbered citations**: "[1]" refers to the first retrieved chunk
- Helps the LLM cite sources
- Lets users verify facts against documents
- Builds trust in the system

**Temperature = 0**: No randomness/creativity
- We want factual answers from documents
- Not creative storytelling

**Explicit "I don't know" instruction**: Critical!
- Prevents hallucinations
- Better to say "not found" than guess
- This is why we retrieve context first (grounds the answer)

### Example System Prompt

```
You are a financial analyst assistant. Your task is to answer questions 
about financial documents using ONLY the provided context.

Rules:
1. Answer only using information from the context below
2. Cite specific sources using [1], [2], [3], etc.
3. If the answer is not in the context, say: "I don't have information about this in the documents."
4. Be concise and factual
```

In [ ]:
# Prompt building functions

SYSTEM_PROMPT = """
You are a financial analyst assistant. Your task is to answer questions 
about financial documents using ONLY the provided context.

Important rules:
1. Answer ONLY using information from the provided context
2. Cite your sources using numbered brackets: [1], [2], [3], etc.
3. If the answer is not found in the context, respond with:
   "I don't have information about this in the provided documents."
4. Be concise, clear, and factual
5. Include relevant metrics and numbers with their context
"""


def build_context_block(chunks: List) -> str:
    """
    Build a context block with numbered citations.
    
    Args:
        chunks: List of retrieved chunks (with .text and .metadata)
        
    Returns:
        Formatted context string ready for prompt
    """
    context_lines = []
    for i, chunk in enumerate(chunks, 1):
        source_info = ""
        if hasattr(chunk, 'metadata'):
            source_file = chunk.metadata.get('source_file', 'Unknown')
            page = chunk.metadata.get('page', 'N/A')
            source_info = f" (Source: {source_file}, Page {page})"
        
        context_lines.append(f"[{i}] {chunk.text}{source_info}")
    
    return "\n\n".join(context_lines)


def build_prompt(question: str, chunks: List) -> str:
    """
    Build the full prompt for answer generation.
    
    Args:
        question: User's question
        chunks: Retrieved chunks
        
    Returns:
        Full prompt string
    """
    context = build_context_block(chunks)
    
    prompt = f"""
{SYSTEM_PROMPT}

CONTEXT:
{context}

QUESTION:
{question}

ANSWER:
"""
    return prompt


# Demo with mock chunks
class MockChunk:
    def __init__(self, text: str, metadata: dict = None, score: float = 0.89):
        self.text = text
        self.metadata = metadata or {}
        self.score = score


mock_chunks = [
    MockChunk(
        "Total revenue for fiscal year 2023 reached $15.2 billion, representing an 18% increase year-over-year.",
        {"source_file": "2023_annual_report.pdf", "page": 5, "doc_type": "annual_report"}
    ),
    MockChunk(
        "This growth was primarily driven by strong performance in our cloud services division, which grew 35% year-over-year.",
        {"source_file": "2023_annual_report.pdf", "page": 6, "doc_type": "annual_report"}
    ),
    MockChunk(
        "Net income for the period was $3.1 billion, with a net margin of 20.4%.",
        {"source_file": "2023_annual_report.pdf", "page": 7, "doc_type": "annual_report"}
    ),
]

print("=" * 60)
print("PROMPT BUILDING")
print("=" * 60)

question = "What was the revenue in 2023 and what drove the growth?"

print(f"\nQuestion: {question}")
print(f"\nRetrieved {len(mock_chunks)} chunks\n")

context = build_context_block(mock_chunks)
print("Context block (as sent to LLM):")
print("-" * 60)
print(context)
print("-" * 60)

print(f"\nFull prompt (first 400 chars):")
print("-" * 60)
full_prompt = build_prompt(question, mock_chunks)
print(full_prompt[:400] + "...")
print("-" * 60)

In [ ]:
# Answer generation class

class RAGGenerator:
    """Generate answers using retrieved context."""
    
    def __init__(self, openai_api_key: str, model: str = "gpt-4o-mini", temperature: float = 0.0):
        self.openai_client = OpenAI(api_key=openai_api_key)
        self.model = model
        self.temperature = temperature
    
    def generate(self, question: str, chunks: List) -> dict:
        """
        Generate an answer using retrieved chunks.
        
        Args:
            question: User question
            chunks: Retrieved chunks
            
        Returns:
            Dict with answer, token counts, and cost
        """
        # Build prompt
        prompt = build_prompt(question, chunks)
        
        # Call LLM
        response = self.openai_client.chat.completions.create(
            model=self.model,
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": prompt}
            ],
            temperature=self.temperature,
        )
        
        # Extract results
        answer = response.choices[0].message.content
        prompt_tokens = response.usage.prompt_tokens
        completion_tokens = response.usage.completion_tokens
        
        # Calculate cost (gpt-4o-mini: $0.15/$0.60 per 1M tokens)
        cost_input = (prompt_tokens / 1_000_000) * 0.15
        cost_output = (completion_tokens / 1_000_000) * 0.60
        total_cost = cost_input + cost_output
        
        return {
            "answer": answer,
            "prompt_tokens": prompt_tokens,
            "completion_tokens": completion_tokens,
            "total_cost_usd": total_cost,
        }


if os.getenv("OPENAI_API_KEY"):
    print("=" * 60)
    print("ANSWER GENERATION WITH RAG (API AVAILABLE)")
    print("=" * 60)
    
    print("""
    generator = RAGGenerator(
        openai_api_key=os.getenv("OPENAI_API_KEY"),
        model="gpt-4o-mini",
        temperature=0.0
    )
    
    result = generator.generate(
        question="What was the revenue growth in 2023?",
        chunks=mock_chunks
    )
    
    print(f"Answer: {result['answer']}")
    print(f"\nTokens: {result['prompt_tokens']} input + {result['completion_tokens']} output")
    print(f"Cost: ${result['total_cost_usd']:.5f}")
    """)
else:
    print("=" * 60)
    print("ANSWER GENERATION WITH RAG (Expected Output)")
    print("=" * 60)
    print("""
    ⚠️  Set OPENAI_API_KEY to run this cell
    
    Expected output:
    
    Answer:
      According to the annual report [1], total revenue for fiscal year 2023 
      reached $15.2 billion, representing an 18% increase year-over-year. 
      The growth was primarily driven by strong performance in the cloud 
      services division [2], which grew 35% year-over-year, significantly 
      outpacing overall company growth.
    
    Tokens: 287 input + 45 output
    Cost: $0.00004 (0.004 cents)
    
    Key metrics:
      - Input: $0.000043 (at $0.15 per 1M tokens)
      - Output: $0.000027 (at $0.60 per 1M tokens)
      - Total: Ultra cheap per query!
    """)

## Section 7: LangChain LCEL Pipeline

### Why LangChain?

Building RAG from scratch teaches you the concepts, but **LangChain** provides:
- **Declarative pipelines**: Define chains as expressions (LCEL = LangChain Expression Language)
- **Built-in components**: Embedding, retrieval, generation already integrated
- **Streaming support**: Stream responses token-by-token
- **Tracing**: LangSmith integration for debugging and monitoring

### Code Comparison

**From scratch** (~50 lines):
```python
# Initialize
retriever = DenseRetriever(...)
retriever.index_chunks(chunks)
chunks = retriever.search(question, top_k=3)
prompt = build_prompt(question, chunks)
result = generator.generate(question, chunks)
```

**LangChain** (~10 lines):
```python
vectorstore = build_vectorstore(chunks, ...)
retriever = get_retriever(vectorstore)
chain = build_rag_chain(retriever)
answer = chain.invoke({"question": question})
```

In [ ]:
# LangChain RAG pipeline (simplified example)

def build_simple_rag_chain(retriever, openai_api_key: str):
    """
    Build a simple RAG chain using LangChain LCEL.
    
    Args:
        retriever: LangChain retriever (from vectorstore)
        openai_api_key: OpenAI API key
        
    Returns:
        Runnable chain
    """
    from langchain_core.runnables import RunnablePassthrough
    from langchain_openai import ChatOpenAI
    from langchain_core.prompts import ChatPromptTemplate
    from langchain_core.output_parsers import StrOutputParser
    
    # Define prompt template
    template = """
You are a financial analyst. Answer the question using ONLY the provided context.
If you can't find the answer, say "I don't have this information."

Context: {context}

Question: {question}

Answer:
"""
    
    prompt = ChatPromptTemplate.from_template(template)
    llm = ChatOpenAI(api_key=openai_api_key, model="gpt-4o-mini", temperature=0)
    
    # Build the chain with LCEL
    chain = (
        {"context": retriever, "question": RunnablePassthrough()}
        | prompt
        | llm
        | StrOutputParser()
    )
    
    return chain


print("=" * 60)
print("LANGCHAIN PIPELINE")
print("=" * 60)

print("""
✓ LangChain RAG in ~15 lines:

1. Build vectorstore from chunks
vectorstore = build_vectorstore(
    chunks=chunks,
    collection_name="financial_2023",
    qdrant_url="http://localhost:6333",
    openai_api_key=os.getenv("OPENAI_API_KEY")
)

2. Create retriever
retriever = get_retriever(vectorstore, top_k=3)

3. Build chain
chain = build_simple_rag_chain(retriever, openai_api_key)

4. Invoke
result = chain.invoke({"question": "What was the revenue?"})
print(result)

✓ The pipe operator (|) chains components together
✓ Much more readable than from-scratch version
✓ Streaming is just: chain.stream({...})
""")

## Section 8: From Scratch vs LangChain Comparison

Both approaches work. Choose based on your needs:

| Aspect | From Scratch | LangChain |
|--------|-------------|----------|
| **Lines of code** | ~100+ | ~20 |
| **Setup time** | ~2 hours | ~30 mins |
| **Learning value** | Very high | Lower (abstracted) |
| **Customization** | Full control | Some constraints |
| **Streaming** | Manual implementation | Built-in (`chain.stream()`) |
| **Debugging** | Custom logging | LangSmith integration |
| **Production ready** | Need more work | Mostly ready |
| **Community** | Solo | Active ecosystem |

### Recommendation

- **Learning**: Start with **from scratch** (Sections 1-6) to understand concepts
- **Projects**: Use **LangChain** for faster development and built-in features
- **Hybrid**: Understand both, combine as needed

### What's Coming Next (Week 4)

We'll upgrade the architecture with advanced techniques:

```
Question 
  ↓
Embed → [Qdrant Search] → Top-20 Chunks (Dense + BM25 Hybrid)
          ↓
      [Reranker] → Top-3 Chunks (Re-scored by relevance)
          ↓
    [Metadata Filter] → Filter by date/company/type
          ↓
      GPT-4o-mini → Answer with Better Citations
```

**New techniques**:
1. **Hybrid search**: Combine dense (semantic) + sparse (BM25 keyword) retrieval
2. **Reranking**: Use a small model to re-score top results (better precision)
3. **Metadata filtering**: Only search within specific documents/dates
4. **Multi-query retrieval**: Ask multiple variations of the question
5. **Query expansion**: Add related terms to improve recall

These techniques improve both accuracy and performance!

## Summary

### You Now Understand:

1. **Embeddings**: Numbers that represent meaning, distances = semantic similarity
2. **OpenAI Embeddings**: text-embedding-3-small (1536 dims, $0.02 per 1M tokens)
3. **Qdrant**: Vector database for efficient semantic search
4. **Dense Retrieval**: Embed question → search → retrieve top-K chunks
5. **Answer Generation**: Pass chunks to LLM with instructions to cite sources
6. **From Scratch**: Build RAG manually to understand each step
7. **LangChain**: Use pre-built components for faster development

### Key Formulas

**Cosine Similarity**:
```
sim(a, b) = (a · b) / (||a|| × ||b||)
```

**Cost Estimation**:
```
embedding_cost = (num_chunks × avg_tokens_per_chunk / 1_000_000) × $0.02
```

**RAG Pipeline**:
```
Question → [Embed] → [Search Qdrant] → [Top-K Chunks] → [LLM + Prompt] → Answer
```

### Next Steps

1. ✓ Complete this notebook (you're reading it!)
2. Set up Qdrant locally with Docker
3. Create a Jupyter notebook with your own financial documents
4. Experiment with different chunk sizes and top-K values
5. Build a simple web UI with Streamlit or Gradio
6. Check out Week 4 for advanced retrieval techniques

---

**Happy RAG building!** 🚀